# 0.1 | DESI Data: Import Main Survey

In the last notebook, we demonstrated a successful hats-import pipeline on two of the fits files in DESI DR1, main survey, dark program.

In this notebook, we will scale this pipeline up to the entire main survey.

In [ ]:
# Imports
import os
from pathlib import Path
import socket
from dask.distributed import Client


## Planning

### Paths

- Set up our paths to target our cosmos users dir
- Get reference to the input catalog
- Also, set up catalog names 

In [ ]:
# Input catalog
input_main_dir = Path("/global/cfs/cdirs/desi/public/dr1/spectro/redux/iron")
input_redshift_cat = input_main_dir / "zcatalog/v1/zall-pix-iron.fits"
input_spectra_dir = input_main_dir / "healpix/"

# Output catalog
username = "olynn"
users_dir = Path("/global/cfs/cdirs/cosmo/users/")
output_main_dir = users_dir / username / "desi_hats"
os.makedirs(output_main_dir, exist_ok=True)

# Catalog name
catalog_name = "desi_dr1_main"

### Catalog Collections

Will need to look into how to write to a collection, esp for catalogs with named programs (light, dark, backup)

In [ ]:
# TODO

### Set up dask with sufficient memory/workers

In [ ]:
# Make a dask client
client = Client( n_workers=8,memory_limit="16GB",)
client

In [ ]:
# Set up dashboard port forwarding instructions for the user. 

# Note that the compute node hostname is needed to set up the ssh tunnel correctly.
# Also, note we set our username in the paths section above.
current_compute_node = socket.gethostname()

print(f"1. In a local terminal, run: ssh -L 8787:{current_compute_node}:8787 {username}@perlmutter.nersc.gov")
print("2. Enter password + OTP")
print(f"3. See Dask Dashboard in browser at: http://localhost:8787/status")

### Setting default columns in the catalog creation

According to the [catalog arguments](https://hats-import.readthedocs.io/en/stable/catalogs/arguments.html) page in the docs:
> you can pass these key-value sets to the import process with the addl_hats_properties argument, and they will appear in the final hats.properties file:  
>  
> `addl_hats_properties={"hats_cols_default": "id, mjd", "obs_regime": "Optical"},`

In [ ]:
# All columns from last notebook's demo:
# TARGETID	TARGET_RA	TARGET_DEC	DESI_TARGET	BGS_TARGET	MWS_TARGET	FLUX_G	FLUX_R	FLUX_Z	FLUX_W1	FLUX_W2	
# FLUX_IVAR_G	FLUX_IVAR_R	FLUX_IVAR_Z	FLUX_IVAR_W1	FLUX_IVAR_W2	EBV	MORPHTYPE	PHOTSYS	MJD	EXPTIME	NIGHT	
# Z	ZERR	ZWARN	DELTACHI2	SPECTYPE	SUBTYPE	spectra_b	spectra_r	spectra_z

# And the defaults I picked in that notebook were:
# 'TARGETID', 'TARGET_RA', 'TARGET_DEC', 'Z', 'SPECTYPE', 'spectra_b', 'spectra_r', 'spectra_z'

# I'll go with that in the defaults I specify for the import pipeline:
addl_hats_properties = {
    "hats_cols_default": "TARGETID, TARGET_RA, TARGET_DEC, Z, SPECTYPE, spectra_b, spectra_r, spectra_z",
}

# May want to specify hats_cols_survey_id while we're at it:
addl_hats_properties["hats_cols_survey_id"] = "TARGETID"

## Run import pipeline

Note that we've broken out our InputReader from the last notebook into its own file.

In [ ]:
# TODO